# Student Performance AI

## Notebook 06 — Model Inference

### Objective

In this notebook we will build an inference pipeline for the trained student performance model.

We will:

- Load the trained model
- Prepare a new student's input
- Apply the same preprocessing used during training
- Generate a prediction
- Interpret the predicted final grade

The goal is to understand how a trained machine learning model can be used by an actual application.

In [1]:
import sys
from pathlib import Path

import torch
import torch.nn as nn
import pandas as pd

In [2]:
project_root = Path.cwd().parent

sys.path.append(str(project_root))

print(project_root)

f:\ML\AI_Engineering\AI-Engineering-Bootcamp\Projects\student-performance-ai


## --Define Model

Use the exact same architecture from Notebook 05.

In [3]:
class StudentPerformanceModel(nn.Module):

    def __init__(self, input_size):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.network(x)

## — Load Model

In [4]:
model_path = (
    project_root
    / "models"
    / "checkpoints"
    / "student_performance_model.pth"
)

print(model_path)
print("Exists:", model_path.exists())

f:\ML\AI_Engineering\AI-Engineering-Bootcamp\Projects\student-performance-ai\models\checkpoints\student_performance_model.pth
Exists: True


In [5]:
input_size = 58

model = StudentPerformanceModel(input_size)

model.load_state_dict(
    torch.load(
        model_path,
        map_location="cpu"
    )
)

model.eval()

print("Model loaded successfully.")

Model loaded successfully.


In [6]:
model.eval()

StudentPerformanceModel(
  (network): Sequential(
    (0): Linear(in_features=58, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=1, bias=True)
  )
)

## Create a New Student

For the first inference test, we'll take one student from the test dataset.

This is useful because we know what the actual result was.

In [7]:
from src.preprocessing import preprocess_data

X_train_tensor, X_test_tensor, y_train_tensor, y_test_tensor = preprocess_data(
    "../data/raw/student-performance.csv"
)

In [8]:
student_features = X_test_tensor[0].unsqueeze(0)

actual_grade = y_test_tensor[0].item()

print("Input shape:", student_features.shape)
print("Actual G3:", actual_grade)

Input shape: torch.Size([1, 58])
Actual G3: 19.0


In [9]:
with torch.no_grad():

    prediction = model(student_features)

predicted_grade = prediction.item()

print("Predicted G3:", predicted_grade)
print("Actual G3:", actual_grade)

Predicted G3: 18.500778198242188
Actual G3: 19.0


In [10]:
error = actual_grade - predicted_grade

print(f"Actual grade    : {actual_grade:.2f}")
print(f"Predicted grade : {predicted_grade:.2f}")
print(f"Prediction error: {error:.2f}")

Actual grade    : 19.00
Predicted grade : 18.50
Prediction error: 0.50


## Prediction FUnction

In [11]:
def predict_student(model, features):
    """
    Generate a student performance prediction.
    """

    model.eval()

    with torch.no_grad():
        prediction = model(features)

    return prediction.item()

In [12]:
prediction = predict_student(
    model,
    student_features
)

print(f"Predicted G3: {prediction:.2f}")

Predicted G3: 18.50


In [13]:
def interpret_grade(grade):

    if grade >= 16:
        return "Excellent"

    elif grade >= 14:
        return "Very Good"

    elif grade >= 10:
        return "Passing"

    else:
        return "At Risk"

In [14]:
print(interpret_grade(predicted_grade))

Excellent


In [15]:
result = {
    "predicted_grade": round(predicted_grade, 2),
    "actual_grade": round(actual_grade, 2),
    "error": round(error, 2),
    "performance_level": interpret_grade(predicted_grade)
}

result

{'predicted_grade': 18.5,
 'actual_grade': 19.0,
 'error': 0.5,
 'performance_level': 'Excellent'}